In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

chat_history = []

def chat(user_input, max_new_tokens=256):
    global chat_history
    chat_history.append({"role": "user", "content": user_input})
    templated_input = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True 
    )
    #分词转为张量
    inputs = tokenizer(templated_input, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
    )
    #解码输出
    reply = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 提取 Assistant 的新回复部分
    response = reply.split("assistant")[-1].replace(":", "").strip()

    # 存入历史记录
    chat_history.append({"role": "assistant", "content": response})

    return response

#测试多轮对话
print("User: Hello, who are you?")
print("Assistant:", chat("Hello, who are you?"))

print("\nUser:What is the capital of China?")
print("Assistant:", chat("What is the capital of China?"))

print("\nUser:What local specialties are there here?")
print("Assistant:", chat("What local specialties are there here?"))

print("\nUser:What famous attractions are there here?")
print("Assistant:", chat("What famous attractions are there here?"))



User: Hello, who are you?
Assistant: designed to help with questions and tasks. How can I assist you today? 😊

User:What is the capital of China?
Assistant: <think>
Okay, the user asked, "What is the capital of China?" I need to recall the correct answer. China's capital is Beijing. I should confirm that there's no confusion with other countries' capitals. Let me make sure there's no other city that's commonly known as the capital. Beijing is the only one I can think of. So, the response should be straightforward and accurate.
</think>

The capital of China is **Beijing**.

User:What local specialties are there here?
Assistant: <think>
Okay, the user just asked about local specialties in China. Let me think. I need to provide a helpful response. China has many cities and regions with unique cuisines. The user might be looking for food options that are popular in their area. Since they asked for local specialties, I should mention the main cities and their famous dishes.

First, I shoul

In [8]:
sentence="I like the class named GenAI,though i don't use transformers before."
tokens=tokenizer.tokenize(sentence)
token_ids=tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token IDs:", token_ids)

Tokens: ['I', 'Ġlike', 'Ġthe', 'Ġclass', 'Ġnamed', 'ĠGen', 'AI', ',', 'though', 'Ġi', 'Ġdon', "'t", 'Ġuse', 'Ġtransformers', 'Ġbefore', '.']
Token IDs: [40, 1075, 279, 536, 6941, 9316, 15469, 11, 4535, 600, 1513, 944, 990, 86870, 1573, 13]


In [23]:
inputs=tokenizer("What is the capital of China?", return_tensors="pt")
outputs=model(**inputs, output_attentions=True)

print(len(outputs.attentions))
print(outputs.attentions[0].shape) #(batch,heads,seq_len,seq_len)
print(outputs.attentions[0])

28
torch.Size([1, 16, 7, 7])
tensor([[[[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [6.6679e-01, 3.3321e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [1.1884e-01, 6.2589e-01, 2.5528e-01, 0.0000e+00, 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [3.2655e-01, 3.1824e-01, 2.7541e-01, 7.9799e-02, 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [5.2546e-02, 3.2668e-01, 2.1563e-01, 5.3827e-04, 4.0460e-01,
           0.0000e+00, 0.0000e+00],
          [2.3109e-01, 2.1535e-01, 1.8818e-01, 2.7136e-02, 2.5345e-01,
           8.4791e-02, 0.0000e+00],
          [2.3675e-01, 1.2154e-01, 2.6517e-01, 4.6190e-06, 1.4123e-01,
           6.9829e-06, 2.3529e-01]],

         [[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
           0.0000e+00, 0.0000e+00],
          [8.9845e-01, 1.0155e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00,
           0.0000e+00, 0.0000e+00],
      